# Baselines e modelos candidatos

**Tarefas:** T2.1, T2.2, T2.4

Objetivo: correr e verificar esta fase e guardar o que o `Projeto_Final.ipynb` lê
(`reports/experiments.csv`, `reports/figures/`). A lógica reutilizável fica em `src/`.
Regras da modelação: [docs/modelacao_base.md](../docs/modelacao_base.md).

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # os caminhos de src/ (data/, reports/, models/) são relativos à raiz

import pandas as pd

from src.models.baselines import baseline_models, candidate_models, simple_models
from src.models.train import run_cv
from src.pipeline import load_model_data

data = load_model_data()

## 3.1 Baselines (T2.1)

A fasquia: tudo o que vier depois tem de as bater. Cada corrida fica em `reports/experiments.csv` (comparação atual) e em `reports/experiments_v1.csv` (histórico, que nunca se limpa).

In [2]:
results = {name: run_cv(name, model, data, notes="T2.1 baseline").mean()
           for name, model in baseline_models().items()}
pd.DataFrame(results).T.round(3)

,pr_auc,roc_auc,recall_at_10,precision_at_10,recall_at_20,precision_at_20,log_loss,brier
random,0.113,0.500,0.096,0.109,0.194,0.110,0.354,0.101
rule_inpatient_ge_1,0.141,0.592,0.147,0.166,0.297,0.168,11.535,0.334
rank_inpatient,0.176,0.607,0.209,0.237,0.341,0.194,2.129,0.104
logreg,0.210,0.664,0.229,0.260,0.385,0.218,0.336,0.097


## 3.1 Variantes da regressão logística

Com pesos por classe (`class_weight="balanced"`) e com regularização L1 (`C=0.1`).

In [3]:
results.update({name: run_cv(name, model, data, notes="T2.1 variante logreg").mean()
                for name, model in simple_models().items()})
pd.DataFrame(results).T.round(3).sort_values("pr_auc", ascending=False)

,pr_auc,roc_auc,recall_at_10,precision_at_10,recall_at_20,precision_at_20,log_loss,brier
logreg_l1,0.211,0.665,0.230,0.260,0.386,0.219,0.336,0.097
logreg,0.210,0.664,0.229,0.260,0.385,0.218,0.336,0.097
logreg_balanced,0.208,0.665,0.226,0.256,0.383,0.217,0.647,0.226
rank_inpatient,0.176,0.607,0.209,0.237,0.341,0.194,2.129,0.104
rule_inpatient_ge_1,0.141,0.592,0.147,0.166,0.297,0.168,11.535,0.334
random,0.113,0.500,0.096,0.109,0.194,0.110,0.354,0.101


## 3.2 Modelos candidatos (T2.2)

Random forest e gradient boosting (`HistGradientBoostingClassifier`, do scikit-learn), com o pré-processamento comum dentro do pipeline e os mesmos 5 folds de D13. Uma configuração razoável de cada; o tuning fica para o `04_tuning_selection`.

In [4]:
results.update({name: run_cv(name, model, data, notes="T2.2 candidato, sem tuning").mean()
                for name, model in candidate_models().items()})
pd.DataFrame(results).T.round(3).sort_values("pr_auc", ascending=False)

,pr_auc,roc_auc,recall_at_10,precision_at_10,recall_at_20,precision_at_20,log_loss,brier
hist_gradient_boosting,0.234,0.677,0.247,0.280,0.400,0.227,0.332,0.095
random_forest,0.221,0.671,0.233,0.265,0.387,0.219,0.335,0.096
logreg_l1,0.211,0.665,0.230,0.260,0.386,0.219,0.336,0.097
logreg,0.210,0.664,0.229,0.260,0.385,0.218,0.336,0.097
logreg_balanced,0.208,0.665,0.226,0.256,0.383,0.217,0.647,0.226
rank_inpatient,0.176,0.607,0.209,0.237,0.341,0.194,2.129,0.104
rule_inpatient_ge_1,0.141,0.592,0.147,0.166,0.297,0.168,11.535,0.334
random,0.113,0.500,0.096,0.109,0.194,0.110,0.354,0.101


## 3.3 Comparação

Média ± desvio dos 5 folds. Uma diferença menor do que o desvio não é uma diferença.

In [5]:
log = pd.read_csv(ROOT / "reports" / "experiments.csv")
val = log[log["dataset"] == "validation"]
val[["date", "model", "pr_auc", "pr_auc_std", "roc_auc", "recall_at_10", "brier", "note"]].sort_values("pr_auc", ascending=False)

,date,model,pr_auc,pr_auc_std,roc_auc,recall_at_10,brier,note
28,2026-11-28T14:35:26,hgb D27: early stopping (150 iterações),0.2360,0.0082,0.6782,0.2471,0.0952,"D27 (28/11, só treino)"
42,2026-11-28T14:41:07,hgb D28: + HbA1c medida × medicação alterada,0.2348,0.0064,0.6763,0.2483,0.0953,"D28 (28/11, só treino)"
58,2026-11-28T15:10:22,hist_gradient_boosting,0.2345,0.0081,0.6769,0.2467,0.0953,"T2.2 candidato, sem tuning"
85,2026-11-28T15:53:00,hist_gradient_boosting,0.2345,0.0081,0.6769,0.2467,0.0953,"T2.2 candidato, sem tuning"
15,2026-11-06T01:57:16,hist_gradient_boosting,0.2345,0.0081,0.6769,0.2467,0.0953,"T2.2 candidato, sem tuning"
21,2026-11-06T01:59:52,hgb ablação: com indicador de registo incompleto,0.2342,0.0064,0.6777,0.2478,0.0953,F2·2 ablação
66,2026-11-28T15:15:53,hgb ablação: com indicador de registo incompleto,0.2342,0.0064,0.6777,0.2478,0.0953,ablação D18
19,2026-11-06T01:59:17,hgb ablação: sem max_glu_serum,0.2341,0.0072,0.6768,0.2483,0.0953,F2·2 ablação
64,2026-11-28T15:13:23,hgb ablação: sem max_glu_serum,0.2341,0.0072,0.6768,0.2483,0.0953,ablação D18
40,2026-11-28T14:40:24,hgb D28: + Charlson (índice + 15 indicadores),0.2336,0.0059,0.6766,0.2450,0.0953,"D28 (28/11, só treino)"
